# Učitavanje i čišćenje podataka. Priprema dataseta.

In [ ]:
lines = open('./data/movie_lines.txt', encoding='utf-8', errors='ignore').read().split('\n')
conversations = open('./data/movie_conversations.txt', encoding='utf-8', errors='ignore').read().split('\n')

In [ ]:
# Svaka linija sadrži: id linije, id osobe, id filma, ime osobe i tekst (tj. dijalog)
lines[30]

'L576 +++$+++ u2 +++$+++ m0 +++$+++ CAMERON +++$+++ Looks like things worked out tonight, huh?'

In [ ]:
len(lines)

304714

In [ ]:
# Svaka konverzacija sadrži: id jedne osobe, id druge osobe, id filma i listu linija
conversations[:10]

["u0 +++$+++ u2 +++$+++ m0 +++$+++ ['L194', 'L195', 'L196', 'L197']",
 "u0 +++$+++ u2 +++$+++ m0 +++$+++ ['L198', 'L199']",
 "u0 +++$+++ u2 +++$+++ m0 +++$+++ ['L200', 'L201', 'L202', 'L203']",
 "u0 +++$+++ u2 +++$+++ m0 +++$+++ ['L204', 'L205', 'L206']",
 "u0 +++$+++ u2 +++$+++ m0 +++$+++ ['L207', 'L208']",
 "u0 +++$+++ u2 +++$+++ m0 +++$+++ ['L271', 'L272', 'L273', 'L274', 'L275']",
 "u0 +++$+++ u2 +++$+++ m0 +++$+++ ['L276', 'L277']",
 "u0 +++$+++ u2 +++$+++ m0 +++$+++ ['L280', 'L281']",
 "u0 +++$+++ u2 +++$+++ m0 +++$+++ ['L363', 'L364']",
 "u0 +++$+++ u2 +++$+++ m0 +++$+++ ['L365', 'L366']"]

In [ ]:
len(conversations)

83098

In [ ]:
# Cilj nam je da napravimo listu dijaloga (konverzacija), stoga ćemo prvo kreirati dictionary koji za id linije vraća tekst (dijalog) te linije
lineID_to_text = {}

for line in lines:
  data = line.split(' +++$+++ ')
  if len(data) == 5:
    lineID_to_text[data[0]] = data[4]


lineID_to_text['L1045']

'They do not!'

In [ ]:
# Sada kreiramo listu dijaloga
dialogues = []

for conversation in conversations:
  data = conversation.split(' +++$+++ ')
  if len(data) == 4:
    line_ids = data[-1][1:-1].replace("'", "").split(', ')
    dialogue = [lineID_to_text[lineID] for lineID in line_ids]
    dialogues.append(dialogue)


dialogues[0]

['Can we make this quick?  Roxanne Korrine and Andrew Barrett are having an incredibly horrendous public break- up on the quad.  Again.',
 "Well, I thought we'd start with pronunciation, if that's okay with you.",
 'Not the hacking and gagging and spitting part.  Please.',
 "Okay... then how 'bout we try out some French cuisine.  Saturday?  Night?"]

In [ ]:
import re

formatted_dialogues = []

def clean_dialogue(text):
    text = re.sub(r"\[.*?\]|\(.*?\)", "", text)  # Remove scene directions
    text = re.sub(r"[^a-zA-Z0-9.,!?']", " ", text)
    return re.sub(r"\s+", " ", text).strip()


for dialogue in dialogues:
    cleaned = [clean_dialogue(line) for line in dialogue]
    formatted_dialogues.append(" <sep> ".join(cleaned))


formatted_dialogues[0]

"Can we make this quick? Roxanne Korrine and Andrew Barrett are having an incredibly horrendous public break up on the quad. Again. <sep> Well, I thought we'd start with pronunciation, if that's okay with you. <sep> Not the hacking and gagging and spitting part. Please. <sep> Okay... then how 'bout we try out some French cuisine. Saturday? Night?"

In [ ]:
formatted_dialogues = formatted_dialogues[:10000]

In [ ]:
with open("movie_dialogues.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(formatted_dialogues))

# Procesiranje dataseta i tokenizacija

In [ ]:
!pip install transformers

In [ ]:
import tensorflow as tf
from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

In [ ]:
tokenizer.add_tokens(["<sep>"])
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
dataset = tf.data.TextLineDataset("movie_dialogues.txt").filter(lambda x: tf.strings.length(x) > 0)

In [ ]:
import numpy as np

dataset = dataset.shuffle(buffer_size=10000, seed=42)

dataset_size = dataset.reduce(np.int64(0), lambda count, _: count + 1).numpy()

train_size = int(0.8 * dataset_size)
val_size = int(0.1 * dataset_size)

train_dataset = dataset.take(train_size)
remaining_dataset = dataset.skip(train_size)
val_dataset = remaining_dataset.take(val_size)
test_dataset = remaining_dataset.skip(val_size)

In [ ]:
MAX_LENGTH = 512

def encode_fn(line):
    text = line.numpy().decode("utf-8")
    encoded = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_tensors="tf",
        return_attention_mask=True
    )
    input_ids = tf.squeeze(encoded["input_ids"], axis=0)
    attention_mask = tf.squeeze(encoded["attention_mask"], axis=0)
    return input_ids, attention_mask, input_ids

def tf_encode_fn(line):
    input_ids, attention_mask, labels = tf.py_function(
        encode_fn, inp=[line], Tout=(tf.int32, tf.int32, tf.int32)
    )
    input_ids.set_shape([MAX_LENGTH])
    attention_mask.set_shape([MAX_LENGTH])
    labels.set_shape([MAX_LENGTH])
    return {"input_ids": input_ids, "attention_mask": attention_mask}, labels

tokenized_train_dataset = train_dataset.map(tf_encode_fn)
tokenized_val_dataset = val_dataset.map(tf_encode_fn)
tokenized_test_dataset = test_dataset.map(tf_encode_fn)

In [ ]:
batch_size = 8
tokenized_train_dataset = tokenized_train_dataset.shuffle(buffer_size=1000, seed=42).batch(batch_size)
tokenized_val_dataset = tokenized_val_dataset.shuffle(buffer_size=1000, seed=42).batch(batch_size)
tokenized_test_dataset = tokenized_test_dataset.shuffle(buffer_size=1000, seed=42).batch(batch_size)

# GPT-2 model

In [ ]:
from transformers import TFGPT2LMHeadModel
tok2 = GPT2Tokenizer.from_pretrained('gpt2')
model = TFGPT2LMHeadModel.from_pretrained("gpt2")
model.resize_token_embeddings(len(tokenizer))

model.summary()

All PyTorch model weights were used when initializing TFGPT2LMHeadModel.

All the weights of TFGPT2LMHeadModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFGPT2LMHeadModel for predictions without further training.


Model: "tfgpt2lm_head_model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 transformer (TFGPT2MainLay  multiple                  124440576 
 er)                                                             
                                                                 
Total params: 124440576 (474.70 MB)
Trainable params: 124440576 (474.70 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=5e-5)
model.compile(optimizer=optimizer)

In [ ]:
history = model.fit(tokenized_train_dataset, epochs=5, validation_data=tokenized_val_dataset)
model.save_pretrained('cdm_gpt2')
tokenizer.save_pretrained('cdm_gpt2')

Epoch 1/5
1000/1000 [==============================] - 1553s 2s/step - loss: 0.6758 - val_loss: 0.3396
Epoch 2/5
1000/1000 [==============================] - 1515s 2s/step - loss: 0.3645 - val_loss: 0.3556
Epoch 3/5
1000/1000 [==============================] - 1514s 2s/step - loss: 0.3506 - val_loss: 0.3321
Epoch 4/5
1000/1000 [==============================] - 1514s 2s/step - loss: 0.3405 - val_loss: 0.3175
Epoch 5/5
1000/1000 [==============================] - 1515s 2s/step - loss: 0.3302 - val_loss: 0.3112


('cdm_gpt2/tokenizer_config.json',
 'cdm_gpt2/special_tokens_map.json',
 'cdm_gpt2/vocab.json',
 'cdm_gpt2/merges.txt',
 'cdm_gpt2/added_tokens.json')

In [ ]:
!zip -r cdm_gpt2.zip cdm_gpt2

  adding: cdm_gpt2/ (stored 0%)
  adding: cdm_gpt2/vocab.json (deflated 68%)
  adding: cdm_gpt2/merges.txt (deflated 53%)
  adding: cdm_gpt2/tf_model.h5 (deflated 7%)
  adding: cdm_gpt2/special_tokens_map.json (deflated 74%)
  adding: cdm_gpt2/config.json (deflated 51%)
  adding: cdm_gpt2/tokenizer_config.json (deflated 64%)
  adding: cdm_gpt2/added_tokens.json (stored 0%)
  adding: cdm_gpt2/generation_config.json (deflated 24%)


In [ ]:
formatted_dialogues[9076]

"C'mon, Jordan. Do the headwork with me. <sep> It's done with, Royce. Let it go. <sep> Someone screwed you over like this, left unanswered charges hanging over your head, and you're not gonna fight back? <sep> I'm tired of fighting back. I just wanted to come home and be safe and have you here and the river there and just forget the rest of the world, okay? <sep> Well, before you crawl off to die, Jordan, give me five minutes of good headwork."

In [ ]:
prompt = "C'mon, Jordan. Do the headwork with me. <sep>"
encoded_prompt = tokenizer(prompt, return_tensors="tf", padding=True)
print("Input IDs:", encoded_prompt["input_ids"].numpy())
print("Attention Mask:", encoded_prompt["attention_mask"].numpy())

# Generate text with adjusted parameters
generated_ids = model.generate(
    encoded_prompt["input_ids"],
    attention_mask=encoded_prompt["attention_mask"],
    max_length=50,
    do_sample=True,
    top_k=50,
    top_p=0.95,
    temperature=1.0,
    num_return_sequences=1
)

# Inspect raw token IDs
print("Generated IDs:", generated_ids.numpy())

# Decode the generated text
generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print("Generated Text:", generated_text)

NameError: name 'tokenizer' is not defined

In [ ]:
tokens = tokenizer.convert_ids_to_tokens([1639,  1607,   262, 22236,  1074,   284,  6594,   262,   850,  6247,   736,    30,
   1148,   326,  3376,    11,   440,     6, 29354,    30,   220, 50257])

In [ ]:
print(tokenizer.convert_ids_to_tokens([1]))

['"']


In [ ]:
formatted_dialogues[0]

"Can we make this quick? Roxanne Korrine and Andrew Barrett are having an incredibly horrendous public break up on the quad. Again. <sep> Well, I thought we'd start with pronunciation, if that's okay with you. <sep> Not the hacking and gagging and spitting part. Please. <sep> Okay... then how 'bout we try out some French cuisine. Saturday? Night?"

In [ ]:
model.summary()

Model: "tfgpt2lm_head_model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 transformer (TFGPT2MainLay  multiple                  124440576 
 er)                                                             
                                                                 
Total params: 124440576 (474.70 MB)
Trainable params: 124440576 (474.70 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
